# Investment Research Assistant
### Multi-Agent System built with Google ADK

Agents: Orchestrator, Researcher, Reviewer/QA — RAG (Vertex AI + FAISS) and Tavily web search, with Hierarchical Delegation, Feedback Loop, and Parallel Execution communication patterns.


## 0. How to use this document

- Open **one** Google Colab notebook. Use **Markdown cells** (not code cells) as section dividers — type a heading like `## Phase 1: Get the Data`, then press the cell type dropdown or `Ctrl+M M` to make it a Markdown cell instead of running it as code. This is how we keep "one notebook, organized in clear sections."
- Work through the phases **in order**. Each phase depends on the one before it.
- Every code block below is meant to be pasted into its **own** Colab cell and run with Shift+Enter.
- When you hit an error: copy the *exact* error message, paste it back to me, and tell me which Step number you were on. We'll fix it together and I'll record the fix in CLAUDE.md.
- **Editing a cell doesn't re-run it.** If you change code in a cell you already ran (e.g. fixing a typo or a model name), that change has no effect until you press Shift+Enter on that cell *again*. Variables in Colab keep whatever value they had from the last time a cell actually executed — not what the cell currently says. If something behaves like an old version of your code, this is the first thing to check.

---

## 1. Mini-glossary (just enough to follow along)

You don't need to memorize these — refer back as needed.

- **Python**: the programming language we're using. Code is just instructions, run top to bottom.
- **Function**: a named, reusable block of code. Defined with `def function_name(inputs):` and produces an output with `return`.
- **Class / OOP**: a "blueprint" for creating objects that bundle data and functions together. Defined with `class ClassName:`. We use this because ADK's `Agent` is a class — you don't need to write your own classes for most of this project, just understand what one looks like when ADK hands you one.
- **Library / package**: pre-written code someone else made, that we `import` and reuse instead of writing from scratch (e.g., ADK itself).
- **API / API key**: a way for our code to talk to another company's service over the internet (e.g., Google's AI models, Tavily's search). The "key" is like a password proving the request is ours.
- **Agent**: in ADK, an object that wraps an AI model (Gemini) plus instructions and a set of tools it's allowed to use.
- **Tool**: a Python function an agent is allowed to call to do something (look something up, calculate something) instead of just generating text.
- **Embedding**: a list of numbers that represents the *meaning* of a piece of text, so a computer can compare how similar two pieces of text are.
- **Vector store (FAISS)**: a searchable database of embeddings — given a question's embedding, it finds the most similar stored embeddings fast.
- **RAG (Retrieval-Augmented Generation)**: look up relevant text first (retrieval), then hand it to the AI model so it answers using that text (generation) instead of guessing from memory.
- **Session / state**: the running memory of a conversation — what's been asked, what's been found, passed between agents.

---

## 2. Prerequisites checklist

Before Phase 0, confirm you have:
- [ ] A Google Cloud project with billing + Vertex AI API enabled (you confirmed this is done)
- [ ] Your **Project ID** (not the project *name* — find it on the Google Cloud Console homepage, top dropdown)
- [ ] A Tavily API key (you confirmed this is done) — looks like `tvly-xxxxxxxx`
- [ ] A free Google account to open Colab at https://colab.research.google.com

---

## PHASE 0: Colab + Cloud Setup

### Step 0.1 — Create the notebook
Go to colab.research.google.com → New Notebook. Rename it (click the title at top) to `investment_research_assistant`.

### Step 0.2 — Install the libraries we need
New code cell:

In [ ]:
!pip install google-adk google-cloud-aiplatform faiss-cpu tavily-python pypdf -q

**What this does:** downloads 5 packages — ADK itself, Google's Vertex AI client (for embeddings), FAISS (vector search), the Tavily client, and `pypdf` (to read PDF text). The `-q` just keeps the output quiet.
**Verify:** no red error text at the bottom. Warnings in yellow/grey are usually fine.

### Step 0.3 — Store your API key safely (Colab Secrets)
Don't paste your Tavily key directly into code. Instead:
1. Click the **key icon (🔑)** in Colab's left sidebar.
2. Click "Add new secret." Name: `TAVILY_API_KEY`. Value: your actual key.
3. Toggle "Notebook access" on for it.

New code cell:

In [ ]:
from google.colab import userdata
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

**Verify:** run it — no error means it found your key. (Don't print it — that would expose it in the notebook output.)

### Step 0.4 — Connect to your Google Cloud project
New code cell:

In [ ]:
from google.colab import auth
auth.authenticate_user()

import vertexai

PROJECT_ID = "your-project-id-here"   # <-- replace with your real Project ID
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Connected to:", PROJECT_ID)

Running this will pop up a Google sign-in flow — follow it and approve access.
**Verify:** you see `Connected to: your-project-id-here` with no error.
**If it fails:** the error message will usually say which API isn't enabled — that tells us exactly what to turn on in the Cloud Console.

### Step 0.5 — Tell ADK to use Vertex AI, not an API key
This step is easy to miss and causes a confusing error later: by default, ADK assumes you're using the **Gemini Developer API** (the AI Studio API-key route), not Vertex AI — even though you just authenticated to Vertex AI in Step 0.4. Those are two separate auth paths, and ADK needs to be told explicitly which one to use.

New code cell:

In [ ]:
import os

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

**Verify:** no output expected — just run it without error. Run this cell in every fresh Colab session, before creating any `Agent`. If you skip it, agent calls fail with `ValueError: No API key was provided...` even though you're correctly authenticated to Vertex AI.

---

## PHASE 1: Get the Data (10-K filings)

We're using Apple, Microsoft, and Tesla's most recent 10-K filings (annual reports) as our knowledge base.

### Step 1.1 — Download the filings
1. Go to https://www.sec.gov/search-filings (the SEC's official EDGAR system — free, no account needed).
2. Search "Apple" (or ticker `AAPL`). Filter results to form type **10-K**. Open the most recent one and download it as a PDF (most filings have a PDF or HTML version you can save/print-to-PDF).
3. Repeat for Microsoft (`MSFT`) and Tesla (`TSLA`).

You should end up with 3 PDFs, named clearly, e.g. `apple_10k.pdf`, `microsoft_10k.pdf`, `tesla_10k.pdf`.

### Step 1.2 — Upload them to Colab
New code cell:

In [ ]:
from google.colab import files
uploaded = files.upload()   # a file picker will appear — select all 3 PDFs

**Verify:** it prints the filename(s) and size(s) after upload.

> Note: Colab's file storage is **temporary** — it's wiped when your runtime disconnects. If you come back to a fresh session later, just re-run the upload step.

---

## PHASE 2: Build the RAG Pipeline

### Step 2.1 — Extract text from the PDFs
New code cell:

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(filepath):
    """Reads a PDF and returns all its text as one big string."""
    reader = PdfReader(filepath)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

filings = {
    "Apple": extract_text_from_pdf("apple_10k.pdf"),
    "Microsoft": extract_text_from_pdf("microsoft_10k.pdf"),
    "Tesla": extract_text_from_pdf("tesla_10k.pdf"),
}

for company, text in filings.items():
    print(f"{company}: {len(text)} characters extracted")

**Concept check:** `def extract_text_from_pdf(filepath):` is a function — reusable code that takes one input (`filepath`) and gives back one output (`full_text`). We call it 3 times, once per company, and store results in a dictionary (`filings`) — a lookup table of `company name -> its text`.
**Verify:** you see 3 lines with character counts in the tens or hundreds of thousands. If a count is suspiciously small (under ~1000), that PDF may be a scanned image rather than real text — tell me and we'll address it.

### Step 2.2 — Chunk the text (semantic chunking)
A 10-K is far too long to hand to an embedding model in one piece (there's a token limit). We split it into overlapping chunks so each chunk is small but still makes sense on its own.

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=150):
    """Splits text into overlapping chunks of roughly chunk_size characters."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap   # step forward, but overlap a bit
    return chunks

all_chunks = []        # will hold every chunk, from every company
chunk_metadata = []    # parallel list: which company each chunk came from

for company, text in filings.items():
    company_chunks = chunk_text(text)
    all_chunks.extend(company_chunks)
    chunk_metadata.extend([{"company": company, "chunk_index": i} for i in range(len(company_chunks))])

print(f"Total chunks created: {len(all_chunks)}")

**Why overlap?** Without it, a sentence that happens to fall right at a chunk boundary gets cut in half and loses meaning. The overlap means nearby chunks share some text, so context isn't lost at the seams.
**Note on "semantic" chunking:** this version splits by a fixed character count, which is simple and works. A more advanced approach splits on actual sentence/paragraph boundaries so chunks never cut mid-sentence. If you want to upgrade to that later, flag it and we'll add it — it's a clean improvement, not a rebuild.
**Verify:** print shows a chunk count in the hundreds, depending on filing length.

### Step 2.3 — Generate embeddings with Vertex AI

In [ ]:
from vertexai.language_models import TextEmbeddingModel
import numpy as np
import time

embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

def embed_texts(texts, batch_size=5):
    """Turns a list of text chunks into a list of embedding vectors, a few at a time."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        results = embedding_model.get_embeddings(batch)
        all_embeddings.extend([r.values for r in results])
        time.sleep(0.5)   # small pause to avoid hitting rate limits
    return all_embeddings

chunk_embeddings = embed_texts(all_chunks)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print("Embeddings shape:", chunk_embeddings.shape)

**Verify:** `Embeddings shape` should print something like `(742, 768)` — the first number is your chunk count (should match Step 2.2), the second is 768 (the size of every `text-embedding-005` vector — this is fixed by the model, not something we choose).
**If it's slow:** this step calls Google's API once per batch of 5 chunks — for a few hundred chunks, expect a few minutes. That's expected, not an error.

### Step 2.4 — Build the FAISS index

In [ ]:
import faiss

dimension = chunk_embeddings.shape[1]   # should be 768
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(chunk_embeddings)

print("Vectors in index:", faiss_index.ntotal)

**Verify:** `Vectors in index` matches your chunk count from Step 2.2.

### Step 2.5 — Test retrieval end-to-end

In [ ]:
def search_filings(query, k=3):
    """Given a question, returns the k most relevant chunks (with their company + text)."""
    query_embedding = embedding_model.get_embeddings([query])[0].values
    query_embedding = np.array([query_embedding]).astype("float32")

    distances, indices = faiss_index.search(query_embedding, k)

    results = []
    for idx in indices[0]:
        results.append({
            "company": chunk_metadata[idx]["company"],
            "text": all_chunks[idx],
        })
    return results

# Try it:
test_results = search_filings("What are the main risk factors?")
for r in test_results:
    print(f"--- {r['company']} ---")
    print(r['text'][:200], "...\n")

**Verify:** you should see 3 chunks of real 10-K text, ideally from the risk-factors section. If results look irrelevant, the chunking or query might need tweaking — tell me what you see and we'll adjust.

**🎉 Checkpoint: your RAG pipeline works.** This alone satisfies the assignment's RAG requirement (PDF extraction ✅, chunking ✅, Vertex AI embeddings ✅, FAISS ✅). Everything from here builds the *agent* layer on top of it.

---

## PHASE 3: Your First ADK Agent (Researcher)

### Step 3.1 — Wrap retrieval as an ADK "tool"
ADK agents call **tools** — plain Python functions with a docstring explaining what they do (the AI model reads that docstring to decide when to use it).

In [ ]:
def faiss_lookup(query: str) -> dict:
    """Searches the company 10-K filings for information relevant to the query.
    Use this for questions about a company's risk factors, financials, business
    description, or anything that would appear in an annual report.

    Args:
        query: the user's question to search for.

    Returns:
        A dictionary with the most relevant filing excerpts.
    """
    results = search_filings(query, k=8)
    return {"status": "success", "results": results}

**Concept check:** the `-> dict` and `query: str` are *type hints* — they tell ADK (and you) what kind of data goes in and out. The triple-quoted text right under `def` is the **docstring** — ADK shows this to the AI model so it knows what the tool does and when to call it. Write docstrings clearly; they matter as much as the code.
**On `k=8`:** a 10-K's risk-factors section typically spans many distinct risks across many chunks. `k=3` (too narrow) was the original value here — it let retrieval get unlucky and miss most of the actual risk-factors content for broad questions. `k=8` casts a wider net. If answers still feel thin, this is the first number to raise further.

### Step 3.2 — Create the Researcher agent

In [ ]:
from google.adk.agents import Agent

researcher_agent = Agent(
    model="gemini-2.5-flash",
    name="researcher_agent",
    description="Finds information about Apple, Microsoft, and Tesla from their 10-K filings.",
    instruction=(
        "You are a financial research assistant. Use the faiss_lookup tool to find "
        "relevant excerpts from company 10-K filings, then answer the user's question "
        "based on what you find. Always mention which company's filing you used."
    ),
    tools=[faiss_lookup],
)

> **If this import fails** with something like `ImportError: cannot import name 'Agent'`, try `from google.adk.agents.llm_agent import Agent` instead — ADK has renamed/relocated things between versions. Whichever one works, tell me and I'll log it in CLAUDE.md so we stop guessing on every future cell.
> **On the model name:** `"gemini-2.5-flash"` is a real Vertex AI publisher model ID, not the Gemini Developer API's `-latest` aliases (like `gemini-flash-latest`) — those resolve on AI Studio's API but 404 on Vertex AI's model catalog, since they're two separate model registries. If `gemini-2.5-flash` itself ever gets retired, the error message will say so explicitly and name its replacement — swap it in and we move on.

### Step 3.3 — Run the agent and see it work

ADK agents are never called directly (`agent(...)`) and don't have a plain `.run()` either — every call has to go through a **Runner**, which manages a conversation session for you. This isn't a workaround, it's how ADK actually works.

Run this once to define a reusable helper (its own cell):

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

async def call_agent(agent, query, user_id="user1", session_id="session1"):
    """Sends one query to an ADK agent and returns its final text reply."""
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=agent.name, user_id=user_id, session_id=session_id)
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)

    user_message = types.Content(role="user", parts=[types.Part(text=query)])

    final_response = ""
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=user_message):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_response = part.text
    return final_response

**Concept check:** `async def` defines a function that can *pause* while waiting on something slow (like a network call to the AI model) instead of freezing everything else. `await` means "pause here until this finishes." Colab cells support `await` directly — no extra setup needed.

Now test it, in a new cell:

In [ ]:
response = await call_agent(researcher_agent, "What are Tesla's main risk factors?")
print(response)

**Verify:** you should get back a real answer about Tesla's risk factors, ideally citing the filing.

**From here on, every agent call in this plan goes through `await call_agent(...)`, never `.run(...)`.** I've corrected the later phases below to match, so you shouldn't hit this particular error again — if you do, it likely means this `call_agent` helper cell didn't run before the one that needs it.

**🎉 Checkpoint:** you have one working ADK agent that intelligently uses RAG. This is the foundation for everything else — Orchestrator and Reviewer/QA are built the same way, just with different tools and instructions.

---

## PHASE 4: Add Web Search (Tavily)

### Step 4.1 — Wrap Tavily as a tool

In [ ]:
from tavily import TavilyClient
import time

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def web_search(query: str) -> dict:
    """Searches the live web for current information — stock prices, recent news,
    analyst commentary, or anything time-sensitive that wouldn't be in a filing.

    Args:
        query: the search query.

    Returns:
        A dictionary with web search results.
    """
    last_error = None
    for attempt in range(3):
        try:
            response = tavily_client.search(query)
            return {"status": "success", "results": response.get("results", [])}
        except Exception as e:
            last_error = e
            time.sleep(2)   # brief pause before retrying — handles transient network blips
    return {"status": "error", "results": [], "error": str(last_error)}

**On the retry loop:** network calls to external APIs (Tavily here) occasionally drop mid-request for reasons that have nothing to do with your code — Colab's network, Tavily's servers, or just bad luck. Without this, a single transient blip crashes the whole cell with a `ConnectionError`. With it, the function quietly retries twice (2 seconds apart) before giving up and returning a clean `"status": "error"` dict instead of crashing — which also means the agent gets something sensible to react to instead of your whole notebook halting.

### Step 4.2 — Give the Researcher agent both tools
This is the heart of the "intelligently choose RAG vs. web search" requirement — we don't write if/else routing logic ourselves; we give the agent both tools and let the model's instructions guide the choice.

In [ ]:
researcher_agent = Agent(
    model="gemini-2.5-flash",
    name="researcher_agent",
    description="Answers investment research questions using filings and live web data.",
    instruction=(
        "You research Apple, Microsoft, and Tesla.\n"
        "- Use faiss_lookup for questions about risk factors, financial statements, "
        "business descriptions, or anything from an annual report.\n"
        "- Use web_search for anything about current stock price, recent news, or "
        "analyst sentiment — anything time-sensitive.\n"
        "- For questions that need both (e.g. 'is the recent stock move justified by "
        "the fundamentals'), use both tools and combine what you find.\n"
        "Always cite which source (filing or web) each piece of information came from."
    ),
    tools=[faiss_lookup, web_search],
)

**Verify:** test with these 3 questions and confirm the agent picks the right tool(s) each time:
- **Filing-only:** `"What does Microsoft's 10-K say about its main business risks?"` (should use `faiss_lookup` only)
- **Live-data only:** `"What is Tesla's current stock price today?"` (should use `web_search` only)
- **Hybrid:** `"Is Apple's recent stock performance consistent with the risks described in its 10-K filing?"` (should use both, and cite both)

Run each with the same `call_agent` helper from Step 3.3:

In [ ]:
response = await call_agent(researcher_agent, "What does Microsoft's 10-K say about its main business risks?")
print(response)

---

## PHASE 5: Orchestrator Agent (Hierarchical Delegation + Session State)

> **Design choice (flagging this since it's not from the assignment file):** ADK supports two ways to coordinate agents: (1) let the model automatically "transfer" control between agents via a `sub_agents` list, or (2) write straightforward Python that calls each agent explicitly. For a first multi-agent system, I'm recommending option 2 — it's predictable and debuggable. Option 1 exists if you want to explore it later; just say so and we'll add it.

### Step 5.1 — Set up session state

In [ ]:
session_state = {
    "conversation_history": [],
    "last_query_type": None,
    "last_results": None,
}

This is a plain Python dictionary acting as shared memory between agents — each agent can read what came before and write what it found.

### Step 5.2 — The Orchestrator

In [ ]:
async def orchestrator(user_query: str) -> str:
    """Receives the user's question, delegates to the Researcher agent,
    and updates session state. This is the system's entry point."""

    session_state["conversation_history"].append({"role": "user", "text": user_query})

    # Hierarchical delegation: Orchestrator -> Researcher
    researcher_response = await call_agent(researcher_agent, user_query)

    session_state["last_results"] = researcher_response
    session_state["conversation_history"].append({"role": "researcher", "text": researcher_response})

    return researcher_response

# Test it:
answer = await orchestrator("What are Apple's risk factors?")
print(answer)

**Verify:** the answer prints, and `session_state["conversation_history"]` now has 2 entries when you check it (`print(session_state["conversation_history"])`).

*(Note: because `orchestrator` uses `await` inside it, it has to be defined with `async def` and called with `await orchestrator(...)` — same pattern as `call_agent` from Step 3.3.)*

---

## PHASE 6: Reviewer/QA Agent (Feedback Loop)

### Step 6.1 — A staleness-checking tool

In [ ]:
from datetime import datetime

def staleness_checker(query: str, used_rag: bool) -> dict:
    """Flags whether a RAG-only answer might be too stale for a time-sensitive query.

    Args:
        query: the original user question.
        used_rag: whether the Researcher's answer came only from filings.

    Returns:
        A dictionary saying whether a web-search top-up is recommended.
    """
    time_sensitive_words = ["today", "now", "current", "latest", "this week", "recent", "stock price"]
    looks_time_sensitive = any(word in query.lower() for word in time_sensitive_words)

    needs_refresh = used_rag and looks_time_sensitive
    return {"needs_refresh": needs_refresh, "checked_at": str(datetime.now())}

### Step 6.2 — The Reviewer/QA agent

In [ ]:
reviewer_agent = Agent(
    model="gemini-2.5-flash",
    name="reviewer_agent",
    description="Reviews Researcher answers for staleness and citation accuracy before they reach the user.",
    instruction=(
        "Review the given answer to the given question. Use staleness_checker to confirm "
        "whether the answer needs a live-data refresh. If it does, say so clearly and "
        "explain what's missing. Otherwise, confirm the answer is ready."
    ),
    tools=[staleness_checker],
)

### Step 6.3 — Wire the feedback loop into the Orchestrator

In [ ]:
async def orchestrator(user_query: str) -> str:
    session_state["conversation_history"].append({"role": "user", "text": user_query})

    researcher_response = await call_agent(researcher_agent, user_query)

    # Feedback loop: Reviewer checks the Researcher's work
    review = await call_agent(reviewer_agent, f"Question: {user_query}\nAnswer: {researcher_response}")

    if "needs a live-data refresh" in review.lower() or "stale" in review.lower():
        # Send it back to the Researcher with a specific instruction
        researcher_response = await call_agent(
            researcher_agent,
            f"{user_query}\n(Note: your previous answer was flagged as potentially stale — "
            f"please also check web_search for the latest information.)"
        )

    session_state["last_results"] = researcher_response
    session_state["conversation_history"].append({"role": "researcher", "text": researcher_response})
    return researcher_response

**Verify:** ask a deliberately time-sensitive question ("What's Tesla's stock doing today, based on the filing?") and confirm the Reviewer triggers a refreshed, web-search-informed answer.

**🎉 Checkpoint:** you now have all 3 required agents and 2 communication patterns (Hierarchical Delegation + Feedback Loop) — the assignment's minimum bar is met. Phase 7 adds a third pattern (Parallel Execution) as a buffer.

---

## PHASE 7: Parallel Execution (for Hybrid Queries)

In [ ]:
import concurrent.futures

async def run_parallel_research(user_query: str) -> str:
    """Runs RAG lookup and web search at the same time for hybrid questions,
    then merges both into one answer."""

    with concurrent.futures.ThreadPoolExecutor() as executor:
        rag_future = executor.submit(faiss_lookup, user_query)
        web_future = executor.submit(web_search, user_query)

        rag_result = rag_future.result()
        web_result = web_future.result()

    merge_prompt = (
        f"Question: {user_query}\n\n"
        f"Filing data found: {rag_result}\n\n"
        f"Web data found: {web_result}\n\n"
        "Combine these into one clear answer, citing filing vs. web sources."
    )
    return await call_agent(researcher_agent, merge_prompt)

**Concept check:** `ThreadPoolExecutor` runs both lookups *at the same time* instead of one after another — that's the "parallel" in Parallel Execution. Without it, we'd wait for RAG to finish, then wait for web search to finish, one after the other. (Note this function is `async` too, since it ends by awaiting `call_agent` — run it with `result = await run_parallel_research("...")`.)
**Verify:** time a hybrid query with this function vs. calling both tools one after another — the parallel version should be noticeably faster.

---

## PHASE 8: Observability

Simple, beginner-friendly logging — not over-engineered:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")
logger = logging.getLogger("investment_assistant")

def log_agent_call(agent_name: str, query: str, duration_seconds: float):
    logger.info(f"agent={agent_name} | query='{query[:50]}' | duration={duration_seconds:.2f}s")

Wrap each agent call with timing, e.g.:

In [ ]:
import time

user_query = "What does Microsoft's 10-K say about its main business risks?"  # or any question you want to time

start = time.time()
researcher_response = await call_agent(researcher_agent, user_query)
log_agent_call("researcher_agent", user_query, time.time() - start)

This satisfies the observability requirement without needing a separate monitoring service — Colab's own output is your log viewer. (If you want a real dashboard later, that's an upgrade, not a requirement.)

---

## PHASE 9: Testing

A few basic checks, run as plain Python (no special test framework needed for a Colab project):

In [ ]:
def test_rag_only_query():
    result = faiss_lookup("What are Apple's risk factors?")
    assert result["status"] == "success"
    assert len(result["results"]) > 0
    print("✅ RAG lookup test passed")

def test_web_search_query():
    result = web_search("Tesla stock price today")
    assert result["status"] == "success"
    print("✅ Web search test passed")

test_rag_only_query()
test_web_search_query()

---